# ZotVision — Auto Labeler
**CLIP + YOLOv8x** zero-shot labeling for firefighter hazard images.

Set your Google Drive image folder path in **cell 3**, run all cells.  
Output: `firefighter_results.json` saved next to your images on Drive.

## 1 · Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless tqdm open-clip-torch pillow

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')

## 3 · Set your image folder path
Point `IMAGE_DIR` at the folder on your Drive that contains the images to label.

In [ ]:
# ── EDIT THIS ──────────────────────────────────────────────────────
IMAGE_DIR = '/content/drive/MyDrive/ZotVision/images'
# ───────────────────────────────────────────────────────────────────

# Output JSON is written into the same folder
import os
OUTPUT_JSON = os.path.join(IMAGE_DIR, 'firefighter_results.json')

print(f'Image dir : {IMAGE_DIR}')
print(f'Output    : {OUTPUT_JSON}')

if not os.path.isdir(IMAGE_DIR):
    raise RuntimeError(f'Directory not found: {IMAGE_DIR}  — check your IMAGE_DIR path above.')

## 4 · Detection config & model loading

In [ ]:
YOLO_CONFIDENCE  = 0.50
YOLO_MIN_BOX_AREA = 0.005

CLIP_MODEL_NAME = 'ViT-L-14'
CLIP_PRETRAINED = 'openai'
CLIP_HAZARD_MARGIN        = 0.03
CLIP_HAZARD_MAX_THRESHOLD = 0.25

HAZARD_PROMPTS = [
    'a photo of fire', 'a photo of flames burning',
    'a photo of a building engulfed in fire', 'a photo of a wildfire with orange flames',
    'a photo of an active fire emergency with visible flames', 'a photo of burning structures',
    'a photo of a house on fire', 'a photo of a car on fire',
    'a photo of flames and smoke in the sky', 'a photo of smoke',
    'a photo of smoke in the air', 'a photo of white smoke rising',
    'a photo of gray smoke haze', 'a photo of thick black smoke',
    'a photo of smoke filling the air', 'a photo of dense smoke billowing from a fire',
    'a photo of a smoky scene', 'a photo of smoke coming from a building',
]
SAFE_PROMPTS = [
    'a normal photo with no fire or smoke', 'a photo of a safe indoor scene',
    'a photo of a landscape with a clear blue sky', 'a photo of a room with no danger',
    'a photo of everyday objects with no emergency',
    'a photo of people going about their day safely',
    'a photo of a sunset with orange sky but no fire', 'a photo of fog or mist with no fire',
    'a photo of clouds in the sky', 'a photo of steam or vapor with no fire',
    'a photo of a city street with no emergency', 'a photo of trees and nature with no fire',
]

In [ ]:
import torch
import open_clip
from ultralytics import YOLO

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

print('Loading YOLOv8x...')
yolo_model = YOLO('yolov8x.pt')

print(f'Loading CLIP ({CLIP_MODEL_NAME})...')
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED
)
clip_model = clip_model.to(DEVICE).eval()
tokenizer  = open_clip.get_tokenizer(CLIP_MODEL_NAME)

print('Models ready.')

## 5 · Detection functions

In [ ]:
from PIL import Image
import numpy as np


def detect_person(img_path):
    results = yolo_model(img_path, verbose=False, conf=YOLO_CONFIDENCE)
    for result in results:
        img_area = result.orig_shape[0] * result.orig_shape[1]
        for box in result.boxes:
            if yolo_model.names[int(box.cls[0].item())] == 'person':
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                if (x2-x1)*(y2-y1) / img_area >= YOLO_MIN_BOX_AREA:
                    return True
    return False


def detect_hazard(img_path):
    image = preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    text  = tokenizer(HAZARD_PROMPTS + SAFE_PROMPTS).to(DEVICE)
    ac    = 'cpu' if DEVICE.type == 'mps' else DEVICE.type
    with torch.no_grad(), torch.amp.autocast(device_type=ac):
        img_f  = clip_model.encode_image(image)
        txt_f  = clip_model.encode_text(text)
        img_f  = img_f / img_f.norm(dim=-1, keepdim=True)
        txt_f  = txt_f / txt_f.norm(dim=-1, keepdim=True)
        sim    = (img_f @ txt_f.T)[0]
    h_sim = sim[:len(HAZARD_PROMPTS)]
    s_sim = sim[len(HAZARD_PROMPTS):]
    return (h_sim.mean().item() > s_sim.mean().item() + CLIP_HAZARD_MARGIN) or \
           (h_sim.max().item() > CLIP_HAZARD_MAX_THRESHOLD)


def label_image(img_path):
    has_person = detect_person(img_path)
    has_hazard = detect_hazard(img_path)
    if has_hazard and has_person: return 'both'
    if has_hazard:                return 'hazard'
    if has_person:                return 'person'
    return 'null'

print('Detection functions defined.')

## 6 · Run labeling & save `firefighter_results.json`

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm

image_paths = sorted(
    [p for ext in ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.bmp')
       for p in Path(IMAGE_DIR).rglob(ext)],
    key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem,
)

if not image_paths:
    raise RuntimeError(f'No images found in {IMAGE_DIR}')
print(f'Found {len(image_paths)} images — labeling...')

stats   = {'hazard': 0, 'person': 0, 'both': 0, 'null': 0}
results = []

for img_path in tqdm(image_paths, desc='Labeling', unit='img'):
    lbl = label_image(str(img_path))
    stats[lbl] += 1
    results.append({'file': img_path.name, 'label': lbl})

output = {
    'image_dir': IMAGE_DIR,
    'total':     len(results),
    'stats':     stats,
    'labels':    results,
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\nDone!')
print(f'  hazard : {stats["hazard"]}')
print(f'  person : {stats["person"]}')
print(f'  both   : {stats["both"]}')
print(f'  null   : {stats["null"]}')
print(f'\nSaved → {OUTPUT_JSON}')